In [ ]:
import h5py
import numpy as np
from tqdm import tqdm
from scipy import ndimage
import os

# 📁 Percorso ai file HDF5 caricati su Kaggle
base_path = "/kaggle/input/hdf5-chunk1" 

# 📁 Directory di output scrivibile su Kaggle
output_dir = "/kaggle/working"

dataset_types = ['train', 'val', 'test']

for dataset_type in dataset_types:
    input_path = os.path.join(base_path, f"gas_and_brake_{dataset_type}_comma_chunk_1_w_imgs.h5py")
    output_path = os.path.join(output_dir, f"filtered_chunk1_{dataset_type}.hdf5")

    print(f"🔄 Processing {dataset_type} from: {input_path}")

    if not os.path.exists(input_path):
        print(f"❌ File not found: {input_path}")
        continue

    with h5py.File(input_path, "r") as h5_file, h5py.File(output_path, "w") as h_out:
        keys = list(h5_file.keys())

        for key in tqdm(keys, desc=f"Processing {dataset_type}"):
            group_in = h5_file[key]

            #if 'desired_dist' not in group_in:
             #   continue

            #desired = np.array(group_in['desired_dist'][()])
            #filtered = ndimage.median_filter(desired, size=12)

            #if (filtered == 0).mean() > 0.2:
             #   continue  # scarta campioni poco informativi

            group_out = h_out.create_group(key)

            for col in group_in.keys():
                dt = np.float32 if col != 'image' else int
                group_out.create_dataset(
                    col,
                    data=group_in[col],
                    compression='gzip',
                    compression_opts=6,
                    chunks=True
                )

    print(f"✅ Salvato in: {output_path}\n")

In [ ]:
import h5py
import os
import numpy as np
from PIL import Image

# Percorso al file hdf5
# 📁 Percorso ai file HDF5 caricati su Kaggle
input_file = "/kaggle/input/hdf5-chunk1/gas_and_brake_test_comma_chunk_1_w_imgs.hdf5"  # Assicurati che il nome del dataset sia giusto

# 📁 Directory di output scrivibile su Kaggle
output_dir = "/kaggle/working"

os.makedirs(output_dir, exist_ok=True)

# Apri il file
with h5py.File(input_file, "r") as h5f:
    for seq_key in h5f.keys():  # per ogni sequenza (es. "2020-03-10--13-47-33_002")
        group = h5f[seq_key]
        images = group['image'][()]  # shape (N, 320, 160, 3)

        seq_dir = os.path.join(output_dir, seq_key)
        os.makedirs(seq_dir, exist_ok=True)

        for i, img in enumerate(images):
            img_pil = Image.fromarray(img.astype(np.uint8))  # da array a immagine
            img_pil.save(os.path.join(seq_dir, f"frame_{i:03}.png"))

        print(f"✅ Salvati {len(images)} frame per: {seq_key}")

In [ ]:
import shutil

# Percorso alla cartella che vuoi scaricare
folder_path = "/kaggle/working/b0c9d2329ad1606b|2018-07-30--13-44-30"

# Sostituisci '|' con '_' per evitare errori nel nome del file zip
safe_name = "b0c9d2329ad1606b_2018-07-30--13-44-30"
output_zip = f"/kaggle/working/{safe_name}.zip"

# Comprimi la cartella
shutil.make_archive(output_zip.replace(".zip", ""), 'zip', folder_path)

In [ ]:
import h5py
import os
import cv2
import numpy as np
from tqdm import tqdm

# === CONFIGURAZIONE ===
input_file = "/kaggle/input/hdf5-chunk1/gas_and_brake_train_comma_chunk_1_w_imgs.hdf5"  # o val/test
output_dir = "/kaggle/working/videos"
os.makedirs(output_dir, exist_ok=True)

# === FUNZIONE PRINCIPALE ===
with h5py.File(input_file, "r") as h5f:
    for seq_key in tqdm(h5f.keys(), desc="Processing sequences"):
        group = h5f[seq_key]

        if 'image' not in group:
            print(f"❌ Nessuna immagine per {seq_key}")
            continue

        images = group['image'][()]  # shape (N, H, W, 3)

        # Verifica che ci siano abbastanza frame
        if len(images) < 2:
            print(f"⚠️ Sequenza troppo corta: {seq_key}")
            continue

        # Imposta parametri video
        height, width = images[0].shape[0], images[0].shape[1]
        video_name = seq_key.replace("|", "_") + ".mp4"  # per evitare errori nel nome file
        video_path = os.path.join(output_dir, video_name)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(video_path, fourcc, 10, (width, height))  # 10 FPS

        for img in images:
            img_bgr = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2BGR)
            out.write(img_bgr)

        out.release()
        print(f"🎥 Video salvato: {video_path}")